In [1]:
!pip -q install --upgrade openai datasets pandas tqdm python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.8/786.8 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 40.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.1 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.1 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.1 which is incompatible.


In [ ]:
# 1) Install exact packages we need
!pip -q install --upgrade openai datasets pandas tqdm

# 2) Imports
import os, json, pathlib, random, uuid
from datetime import datetime
from typing import Dict, Any, List

import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from openai import OpenAI  # used for BOTH OpenAI and DeepInfra (DeepInfra is OpenAI-compatible via base_url)

# 3) 🔑 Paste your API keys directly here (remember to redact before committing to GitHub!)
OPENAI_API_KEY    = ""
DEEPINFRA_API_KEY = ""  # no prefix check; any non-empty string is accepted

assert isinstance(OPENAI_API_KEY, str) and len(OPENAI_API_KEY) > 0, "OpenAI API key is empty"
assert isinstance(DEEPINFRA_API_KEY, str) and len(DEEPINFRA_API_KEY) > 0, "DeepInfra API key is empty"

# 4) Run folder (everything we save goes here)
RUN_TAG = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
RUN_DIR = pathlib.Path(f"runs/{RUN_TAG}").resolve()
(RUN_DIR / "preds").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "tables").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "plots").mkdir(parents=True, exist_ok=True)

# 5) Global constants (we’ll keep these fixed through the run)
N_PER_DEPTH = 10          # number of items per depth for the FULL run (change to 2 only if you ever want a quick smoke test)
RANDOM_SEED = 42

# Decoding budgets
DIRECT_MAX = 32           # Direct mode: short final answer only
REASON_MAX = 256          # Reasoning mode: allow rationale + answer

# Budgets for novel protocols (DCAB + AV‑vs‑TA)
BUDGETS = [64, 128, 256]  # you can trim to [64,128] if you need to save costs further

# 6) Model registry (providers, prices, feature flags)
MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {
    # -------------------- DeepInfra (OpenAI-compatible Chat Completions) --------------------
    "llama31_8b": {
        "provider": "deepinfra",
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "price_in": 0.03,    # $ / 1M input tokens
        "price_out": 0.05,   # $ / 1M output tokens
        "supports_reasoning_effort": False,   # use CoT prefix for "reasoning"
        "uses_max_output_tokens": False       # Chat Completions uses 'max_tokens'
    },
    "qwen25_7b": {
        "provider": "deepinfra",
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "price_in": 0.04,
        "price_out": 0.10,
        "supports_reasoning_effort": False,
        "uses_max_output_tokens": False
    },
    "deepseek_r1d_qwen32b": {
        "provider": "deepinfra",
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
        "price_in": 0.075,
        "price_out": 0.15,
        "supports_reasoning_effort": False,   # plain CoT prefix
        "uses_max_output_tokens": False
    },
    "glm45_air": {
        "provider": "deepinfra",
        "model": "zai-org/GLM-4.5-Air",
        "price_in": 0.20,
        "price_out": 1.10,
        "supports_reasoning_effort": True,    # pass 'reasoning_effort' for this model
        "uses_max_output_tokens": False
    },

    # -------------------- OpenAI (Responses API for GPT‑5) --------------------
    "gpt5": {
        "provider": "openai",
        "model": "gpt-5",
        "price_in": 1.25,    # adjust if your account differs
        "price_out": 10.0,
        "supports_reasoning_effort": True,    # reasoning={"effort": "minimal"|"low"|"medium"|"high"}
        "uses_max_output_tokens": True        # Responses API uses 'max_output_tokens'
    }
}

# 7) Create API clients (no network call yet)
openai_client = OpenAI(api_key=OPENAI_API_KEY)  # OpenAI (GPT‑5) Responses API
deepinfra_client = OpenAI(api_key=DEEPINFRA_API_KEY, base_url="https://api.deepinfra.com/v1/openai")

# Utility: mask keys when printing
def _mask(k: str) -> str:
    if not k: return "<empty>"
    if len(k) <= 8: return "*" * len(k)
    return k[:4] + "*" * (len(k)-8) + k[-4:]

# 8) Persist a minimal run config (NO KEYS)
run_config = {
    "run_tag": RUN_TAG,
    "constants": {
        "N_PER_DEPTH": N_PER_DEPTH,
        "DIRECT_MAX": DIRECT_MAX,
        "REASON_MAX": REASON_MAX,
        "BUDGETS": BUDGETS,
        "RANDOM_SEED": RANDOM_SEED
    },
    "models": {k: {
        "provider": v["provider"],
        "model": v["model"],
        "price_in": v["price_in"],
        "price_out": v["price_out"],
        "supports_reasoning_effort": v["supports_reasoning_effort"],
        "uses_max_output_tokens": v["uses_max_output_tokens"],
    } for k, v in MODEL_REGISTRY.items()}
}
with open(RUN_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

# 9) Human-readable summary
print("✅ Step 1 ready\n")
print("Keys loaded (masked):")
print("  OpenAI   :", _mask(OPENAI_API_KEY))
print("  DeepInfra:", _mask(DEEPINFRA_API_KEY))
print("\nClients initialized:")
print("  OpenAI client has 'responses':", hasattr(openai_client, "responses"))
print("  DeepInfra client Chat Completions available:",
      hasattr(deepinfra_client, "chat") and hasattr(deepinfra_client.chat, "completions"))
print("\nRun directory:", RUN_DIR)

summary_rows = []
for key, cfg in MODEL_REGISTRY.items():
    summary_rows.append({
        "model_key": key,
        "provider": cfg["provider"],
        "model": cfg["model"],
        "$/M IN": cfg["price_in"],
        "$/M OUT": cfg["price_out"],
        "reasoning_effort?": cfg["supports_reasoning_effort"],
        "uses_max_output_tokens?": cfg["uses_max_output_tokens"]
    })
display(pd.DataFrame(summary_rows))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.8/786.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.1 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.1 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.1 which is incompatible.
✅ Step 1 ready

Keys loaded (masked):
  OpenAI   : sk-p************************************************************************************************************************************************************5YoA
  DeepInfra: gbsu************************1Xje

Clients initialized:
  Open

,model_key,provider,model,$/M IN,$/M OUT,reasoning_effort?,uses_max_output_tokens?
0,llama31_8b,deepinfra,meta-llama/Meta-Llama-3.1-8B-Instruct,0.030,0.05,False,False
1,qwen25_7b,deepinfra,Qwen/Qwen2.5-7B-Instruct,0.040,0.10,False,False
2,deepseek_r1d_qwen32b,deepinfra,deepseek-ai/DeepSeek-R1-Distill-Qwen-32B,0.075,0.15,False,False
3,glm45_air,deepinfra,zai-org/GLM-4.5-Air,0.200,1.10,True,False
4,gpt5,openai,gpt-5,1.250,10.00,True,True


In [6]:
# STEP 2 — Datasets & Sampling via Parquet (works with HF Datasets >= 3.0)
# Uses from Step 1: N_PER_DEPTH, RANDOM_SEED, RUN_DIR
# Produces:
#   - ITEMS (list of examples with a unified schema)
#   - runs/<ts>/subset_ids.json
#   - runs/<ts>/tables/items_count_by_depth.csv
#   - runs/<ts>/tables/items_preview.csv

import re, hashlib, json, random, pathlib, requests
from collections import defaultdict
import pandas as pd
from datasets import load_dataset

random.seed(RANDOM_SEED)

# -------------------------- Helper: list Parquet URLs --------------------------
# Uses the Dataset Viewer API: https://datasets-server.huggingface.co/parquet
def hf_parquet_urls(dataset:str, config:str, split:str):
    """
    Return a list of Parquet file URLs for (dataset, config, split).
    Example:
      hf_parquet_urls("CLUTRR/v1", "gen_train234_test2to10", "train")
    """
    base = "https://datasets-server.huggingface.co/parquet"
    r = requests.get(base, params={"dataset": dataset, "config": config, "split": split}, timeout=30)
    r.raise_for_status()
    js = r.json()
    files = js.get("parquet_files", [])
    if not files:
        raise RuntimeError(f"No parquet files listed for {dataset} [{config}/{split}]. "
                           f"Viewer may be warming; try split='validation' or 'test'.")
    return [f["url"] for f in files]

def load_parquet_dataset(urls, split_name="train"):
    """
    Load a dataset from a list of Parquet URLs using HF Datasets 'parquet' builder.
    """
    data_files = {split_name: urls}
    ds = load_dataset("parquet", data_files=data_files, split=split_name, trust_remote_code=False)
    return ds

# ----------------------------- CLUTRR (Parquet) -------------------------------
# CLUTRR has subsets (“builder configs”); we use the common one with train/val/test.
CLUTRR_DATASET = "CLUTRR/v1"
CLUTRR_CONFIG  = "gen_train234_test2to10"   # see dataset page subsets
# Label set (from CLUTRR card)
CLUTRR_LABELS = [
    "aunt","son-in-law","grandfather","brother","sister","father","mother",
    "grandmother","uncle","daughter-in-law","grandson","granddaughter",
    "father-in-law","mother-in-law","nephew","son","daughter","niece",
    "husband","wife","sister-in-law"
]
_depth_pat = re.compile(r"\.(\d+)$")  # matches 'task_1.3' -> 3

def _clutrr_depth_of(row):
    """
    Depth k from CLUTRR row.
    Prefer 'task_name' suffix (task_1.k), fallback to f_comb length or '->' arrows.
    """
    tn = str(row.get("task_name", ""))
    m = _depth_pat.search(tn)
    if m:
        try:
            return int(m.group(1))
        except Exception:
            pass
    f = row.get("f_comb", None)
    if isinstance(f, str) and f:
        return f.count("-") + 1
    if isinstance(f, list) and len(f) > 0:
        return len(f)
    for k in ("path","solution","proof_state"):
        val = row.get(k)
        if isinstance(val, str):
            cnt = val.count("->")
            if cnt > 0:
                return cnt + 1
    return None

def _clutrr_target_text(row):
    """
    Gold label: use 'target_text' if present; else map numeric 'target' index to text.
    """
    ttxt = row.get("target_text")
    if ttxt:
        return str(ttxt)
    t = row.get("target")
    try:
        idx = int(str(t))
        if 0 <= idx < len(CLUTRR_LABELS):
            return CLUTRR_LABELS[idx]
    except Exception:
        pass
    return str(t) if t is not None else ""

def load_clutrr(depths=(2,3,4,5,6), n_per_depth=10, seed=42):
    # Try train first; if no parquet yet (rare), fall back to validation then test
    for split in ["train", "validation", "test"]:
        try:
            urls = hf_parquet_urls(CLUTRR_DATASET, CLUTRR_CONFIG, split)
            clutrr_ds = load_parquet_dataset(urls, split_name=split)
            break
        except Exception as e:
            last_err = e
            clutrr_ds = None
    if clutrr_ds is None:
        raise RuntimeError(f"Failed to load CLUTRR parquet: {last_err}")

    rng = random.Random(seed)
    items = []
    # Convert to plain dicts for easier access
    rows = [clutrr_ds[i] for i in range(len(clutrr_ds))]

    for k in depths:
        subset = [ex for ex in rows if _clutrr_depth_of(ex) == k]
        rng.shuffle(subset)
        take = subset[:n_per_depth]
        if len(take) < n_per_depth:
            print(f"[WARN] CLUTRR: requested {n_per_depth} at k={k}, got {len(take)} from split '{split}'.")
        for ex in take:
            story = ex.get("story") or ex.get("clean_story") or ""
            query = ex.get("query") or ""
            label = _clutrr_target_text(ex)
            base_id = ex.get("id") or ex.get("uid") \
                      or hashlib.sha1((str(story) + "||" + str(query)).encode("utf-8")).hexdigest()[:16]
            items.append({
                "dataset": "clutrr",
                "depth": k,
                "id": f"clutrr:{base_id}",
                "input": str(story),
                "query": str(query),
                "choices": CLUTRR_LABELS,
                "answer": str(label)
            })
    return items

# --------------------------- ProofWriter (Parquet) ----------------------------
PW_DATASET = "tasksource/proofwriter"
PW_CONFIG  = "default"     # ProofWriter exposes a 'default' config with QDep
def _load_pw_parquet(split="test"):
    urls = hf_parquet_urls(PW_DATASET, PW_CONFIG, split)
    return load_parquet_dataset(urls, split_name=split), split

def _detect_pw_schema(ds):
    cols = set(ds.column_names)
    def pick(keys): return next((k for k in keys if k in cols), None)
    return {
        "theory":     pick(["theory","context","facts","paragraph"]),
        "hypothesis": pick(["hypothesis","question","query"]),
        "answer":     pick(["answer","label","ans"]),
        "depth_key":  pick(["QDep","depth","proof_depth","depth_level","qdep"])
    }

def _normalize_pw_label(val):
    if isinstance(val, bool): return "True" if val else "False"
    s = str(val).strip().lower()
    if s in {"true","t","yes","y","entailed","entailment","entails","1"}: return "True"
    if s in {"false","f","no","n","contradiction","not entailed","0"}:     return "False"
    return "True" if s == "yes" else ("False" if s == "no" else str(val))

def load_proofwriter(depths=(1,2,3,4,5), n_per_depth=10, seed=42):
    # Prefer test; if viewer is warming, fall back to validation
    last_err = None
    for split in ["test","validation","train"]:
        try:
            ds, real_split = _load_pw_parquet(split)
            break
        except Exception as e:
            last_err = e
            ds = None
    if ds is None:
        raise RuntimeError(f"Failed to load ProofWriter parquet: {last_err}")

    schema = _detect_pw_schema(ds)
    if not schema["theory"] or not schema["hypothesis"] or not schema["answer"] or not schema["depth_key"]:
        raise RuntimeError(f"ProofWriter schema detection failed on {PW_DATASET}/{real_split}. "
                           f"Columns: {ds.column_names}")

    rng = random.Random(seed)
    items = []
    # Materialize to Python dicts for filtering
    rows = [ds[i] for i in range(len(ds))]
    depth_key = schema["depth_key"]

    for d in depths:
        subset = [ex for ex in rows if str(ex.get(depth_key,"")).strip().isdigit() and int(ex[depth_key]) == d]
        if not subset:
            subset = [ex for ex in rows if str(ex.get(depth_key,"")).strip() == str(d)]
        rng.shuffle(subset)
        take = subset[:n_per_depth]
        if len(take) < n_per_depth:
            print(f"[WARN] ProofWriter: requested {n_per_depth} at D={d}, got {len(take)} from split '{real_split}'.")
        for ex in take:
            theory = str(ex.get(schema["theory"], ""))
            hyp    = str(ex.get(schema["hypothesis"], ""))
            label  = _normalize_pw_label(ex.get(schema["answer"], ""))
            base_id = hashlib.sha1((theory + "||" + hyp).encode("utf-8")).hexdigest()[:16]
            items.append({
                "dataset": "proofwriter",
                "depth": d,
                "id": f"proof:{base_id}",
                "input": theory,
                "query": hyp,
                "choices": ["True","False"],
                "answer": label
            })
    return items

# --------------------------- LOAD & SAVE SUBSET -------------------------------
clutrr_items = load_clutrr(depths=(2,3,4,5,6), n_per_depth=N_PER_DEPTH, seed=RANDOM_SEED)
proof_items  = load_proofwriter(depths=(1,2,3,4,5), n_per_depth=N_PER_DEPTH, seed=RANDOM_SEED)
ITEMS = clutrr_items + proof_items

# Save IDs for reproducibility
(RUN_DIR / "tables").mkdir(exist_ok=True, parents=True)
subset_ids = [{"id": ex["id"], "dataset": ex["dataset"], "depth": ex["depth"]} for ex in ITEMS]
with open(RUN_DIR / "subset_ids.json", "w") as f:
    json.dump(subset_ids, f, indent=2)

# Summaries & previews
df_items = pd.DataFrame([{"dataset": ex["dataset"], "depth": ex["depth"]} for ex in ITEMS])
counts = (df_items.value_counts(["dataset","depth"]).sort_index().reset_index(name="n"))
counts.to_csv(RUN_DIR / "tables" / "items_count_by_depth.csv", index=False)

preview_rows = []
seen = defaultdict(int)
for ex in ITEMS:
    key = (ex["dataset"], ex["depth"])
    if seen[key] < 2:
        preview_rows.append({
            "dataset": ex["dataset"],
            "depth": ex["depth"],
            "id": ex["id"],
            "input_snippet": (ex["input"][:120] + "…") if len(ex["input"]) > 120 else ex["input"],
            "query": ex["query"],
            "gold": ex["answer"]
        })
        seen[key] += 1

preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(RUN_DIR / "tables" / "items_preview.csv", index=False)

print("✅ STEP 2 complete.")
print(f"Total items: {len(ITEMS)}  (CLUTRR: {len(clutrr_items)}, ProofWriter: {len(proof_items)})")
print("Counts by dataset/depth:")
display(counts)
print("Preview (first 2/examples per dataset-depth):")
display(preview_df)


0000.parquet:   0%|          | 0.00/598k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/3.69M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/938k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

default/test/0000.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

✅ STEP 2 complete.
Total items: 100  (CLUTRR: 50, ProofWriter: 50)
Counts by dataset/depth:


,dataset,depth,n
0,clutrr,2,10
1,clutrr,3,10
2,clutrr,4,10
3,clutrr,5,10
4,clutrr,6,10
5,proofwriter,1,10
6,proofwriter,2,10
7,proofwriter,3,10
8,proofwriter,4,10
9,proofwriter,5,10


Preview (first 2/examples per dataset-depth):


,dataset,depth,id,input_snippet,query,gold
0,clutrr,2,clutrr:d4ff2c69-eaa9-4ba0-9f9f-33579afd323f,[Ashley]'s son [Nicholas] and son [George] wen...,"('Nicholas', 'George')",brother
1,clutrr,2,clutrr:2896c293-fe1c-4554-8faa-43d0933a84d0,[Jennifer] always loved when her grandmother [...,"('Natalia', 'Ashley')",grandmother
2,clutrr,3,clutrr:24f02008-e43a-480b-814f-3c5bca945980,[Wesley] and his daughter [Martha] spent Fathe...,"('Lena', 'Pamela')",daughter
3,clutrr,3,clutrr:0634c05e-fc36-4f1b-85f5-ca0b3ce34990,"[Francisco] needed his brother, [Wesley], to h...","('Francisco', 'James')",father
4,clutrr,4,clutrr:cee2db10-8a67-4cae-a3b7-02ef3ba79f9f,[Nicole] misses her father [Louis] when she is...,"('Nicole', 'Harold')",grandfather
5,clutrr,4,clutrr:6b7720df-f16a-4fa6-81dd-b8668bc506cc,[Bobby] and is making a special card for his g...,"('Antonio', 'Allan')",nephew
6,clutrr,5,clutrr:dc0b42ee-b6fe-4746-86e6-b0fd8a10535a,[Milton] invited his father [Jose] and his bro...,"('Jose', 'Adeline')",granddaughter
7,clutrr,5,clutrr:5131260d-b36f-4009-89f7-1c94e325d132,[Guadalupe] had picked her daughter [Margarett...,"('Guadalupe', 'Gilbert')",nephew
8,clutrr,6,clutrr:78e69cbf-0f3b-48fd-a8ed-3fc2382c9d71,[Katherine] and her daughter [Jon] like to loo...,"('Clifton', 'Ronald')",nephew
9,clutrr,6,clutrr:76ff4690-db3d-44e9-af0b-752d39aa84d5,[Erica] loves cooking for her son. His name is...,"('Dorothy', 'Ross')",nephew


In [7]:
# STEP 3 — Prompts, parsers, unified callers
# Relies on Step 1 variables: MODEL_REGISTRY, DIRECT_MAX, REASON_MAX, BUDGETS,
# and clients: openai_client (OpenAI/Responses), deepinfra_client (Chat Completions)

import re, math, time, random
from typing import Dict, Any, List, Optional, Tuple

# ----------------------------- Prompt templates ------------------------------

# CLUTRR (kinship)
CLUTRR_DIRECT_TMPL = """Story:
{story}

Question:
{question}

Answer with ONE label only from this set:
{labels}

Return just the label (no explanation).
"""

CLUTRR_COT_PREFIX = "Let's think step by step.\n\n"

# ProofWriter (logical entailment)
PW_DIRECT_TMPL = """Premises:
{theory}

Hypothesis:
{hyp}

Is the hypothesis entailed by the premises?
Answer exactly one of: True or False.
Return just the single word True or False.
"""

PW_COT_PREFIX = "Let's think step by step.\n\n"

# DCAB (Depth Calibration & Adaptive Budgeting)
DCAB_DEPTH_TMPL = """You will see a reasoning problem. Estimate the MINIMUM number of reasoning hops (an integer 1..6) required to answer it.
Output just the integer (no words).

Problem:
{body}
"""

DCAB_BUDGET_TMPL = """Previously you estimated the required depth as D = {d_hat}.
Choose a THINKING BUDGET from this set: {budget_set}.
Output just ONE number from that set (no words).
"""

# AV-Verify (Answer-then-Verify)
AV_VERIFY_TMPL = """You are given a problem and a PROPOSED ANSWER. Your task is to VERIFY it using only the provided content.
If the proposed answer is wrong or inconsistent, OUTPUT the corrected final label.
Otherwise, OUTPUT the same label. Return ONLY the final label (no explanation).

Problem:
{body}

Proposed answer: {proposed}

Return only the final label (no extra text).
"""


# ----------------------------- Prompt builders -------------------------------

def build_problem_body(ex: Dict[str, Any]) -> str:
    """Returns a compact textual body representing the example, for DCAB/Verify."""
    if ex["dataset"] == "clutrr":
        return f"Story:\n{ex['input']}\n\nQuestion:\n{ex['query']}\n\nAllowed labels: {', '.join(ex['choices'])}"
    else:
        return f"Premises:\n{ex['input']}\n\nHypothesis:\n{ex['query']}\n\nAllowed answers: True, False"

def build_prompt(ex: Dict[str, Any],
                 protocol: str,
                 *,
                 d_hat: Optional[int] = None,
                 b_hat: Optional[int] = None,
                 budgets: Optional[List[int]] = None,
                 proposed: Optional[str] = None,
                 use_cot_prefix: bool = False) -> str:
    """
    protocol ∈ {"direct","reasoning","dcab_depth","dcab_budget","verify"}
    - For 'reasoning', set use_cot_prefix=True for models WITHOUT a native reasoning toggle.
    - For 'dcab_budget', pass d_hat and budgets.
    - For 'verify', pass 'proposed' (the initial label).
    """
    if protocol == "direct":
        if ex["dataset"] == "clutrr":
            return CLUTRR_DIRECT_TMPL.format(
                story=ex["input"],
                question=ex["query"],
                labels=", ".join(ex["choices"])
            )
        else:
            return PW_DIRECT_TMPL.format(
                theory=ex["input"],
                hyp=ex["query"]
            )

    elif protocol == "reasoning":
        core = build_prompt(ex, "direct")
        if use_cot_prefix:
            if ex["dataset"] == "clutrr":
                return CLUTRR_COT_PREFIX + core
            else:
                return PW_COT_PREFIX + core
        else:
            # Provider-native reasoning mode: keep text identical to direct
            return core

    elif protocol == "dcab_depth":
        body = build_problem_body(ex)
        return DCAB_DEPTH_TMPL.format(body=body)

    elif protocol == "dcab_budget":
        assert d_hat is not None, "d_hat is required for dcab_budget"
        bset = budgets or BUDGETS
        body = DCAB_BUDGET_TMPL.format(d_hat=d_hat, budget_set=", ".join(str(b) for b in bset))
        return body

    elif protocol == "verify":
        assert proposed is not None, "proposed label is required for verify"
        body = build_problem_body(ex)
        return AV_VERIFY_TMPL.format(body=body, proposed=proposed)

    else:
        raise ValueError(f"Unknown protocol: {protocol}")


# ------------------------------ Label parsers --------------------------------

def extract_label(text: str, choices: List[str]) -> Optional[str]:
    """
    Heuristic extractor for final labels.
    - For CLUTRR: whole-word match against provided label set.
    - For ProofWriter: look for True/False tokens.
    Returns None if no confident match.
    """
    if not text:
        return None
    t = " " + text.strip().lower() + " "

    # ProofWriter first (True/False)
    if "true" in t and not "false" in t:
        return "True"
    if "false" in t and not "true" in t:
        return "False"

    # CLUTRR labels (whole-word)
    best = None
    best_pos = 10**9
    for lab in choices:
        pat = f" {lab.lower()} "
        pos = t.find(pat)
        if pos != -1 and pos < best_pos:
            best_pos = pos
            best = lab
    return best

UNCERTAINTY_PAT = re.compile(r"\b(i\s*(am|m)\s*not\s*sure|unsure|uncertain|don['’]t\s*know|cannot\s*(decide|answer)|no\s*idea)\b", re.I)

def contains_uncertainty(text: str) -> bool:
    if not text: return False
    return bool(UNCERTAINTY_PAT.search(text))


# ------------------------------- Cost utils ----------------------------------

def estimate_cost_usd(prompt_tokens: int, completion_tokens: int, price_in: float, price_out: float) -> float:
    return round((prompt_tokens/1e6)*price_in + (completion_tokens/1e6)*price_out, 6)


# -------- Mapping between budgets (tokens) and provider effort levels ---------

def map_budget_to_effort(b: int) -> str:
    """
    Map token budgets to qualitative effort levels for providers that expose it.
    """
    if b <= 64:   return "low"
    if b <= 160:  return "medium"
    return "high"


# ---------------------------- Unified model caller ---------------------------

def _extract_usage_openai(resp) -> Tuple[int,int]:
    """
    Handle usage for OpenAI Responses API (fields may vary by SDK version).
    Returns (prompt_tokens, completion_tokens).
    """
    prompt_tok = 0
    completion_tok = 0
    usage = getattr(resp, "usage", None)
    if usage:
        # Try attributes then dict-style
        prompt_tok = getattr(usage, "input_tokens", getattr(usage, "prompt_tokens", 0)) or 0
        completion_tok = getattr(usage, "output_tokens", getattr(usage, "completion_tokens", 0)) or 0
    return int(prompt_tok), int(completion_tok)

def _extract_text_openai(resp) -> str:
    """
    Extract text from Responses API output. Tries output_text, else iterates 'output' items.
    """
    # Newer SDKs
    txt = getattr(resp, "output_text", None)
    if isinstance(txt, str) and txt.strip():
        return txt

    # Fallback: iterate items
    out = []
    output = getattr(resp, "output", None)
    if output:
        try:
            for item in output:
                content = getattr(item, "content", None)
                if content:
                    for c in content:
                        t = getattr(c, "text", None)
                        if t:
                            out.append(str(t))
        except Exception:
            pass
    return "\n".join(out).strip()

def _extract_usage_chat(resp) -> Tuple[int,int]:
    """
    DeepInfra Chat Completions usage extraction.
    """
    usage = getattr(resp, "usage", None)
    if usage:
        pt = getattr(usage, "prompt_tokens", 0) or 0
        ct = getattr(usage, "completion_tokens", 0) or 0
        return int(pt), int(ct)
    return 0, 0

def _extract_text_chat(resp) -> str:
    try:
        return str(resp.choices[0].message.content)
    except Exception:
        return ""

def call_model(model_key: str,
               prompt: str,
               *,
               mode: str = "direct",
               max_output_tokens: int = 32,
               effort: Optional[str] = None,
               temperature: float = 0.0,
               top_p: float = 1.0,
               retries: int = 2,
               retry_backoff: float = 1.5) -> Dict[str, Any]:
    """
    Unified caller for both providers.
    - For OpenAI (GPT‑5): uses Responses API with 'max_output_tokens' and optional 'reasoning={"effort": ...}'.
    - For DeepInfra: uses Chat Completions with 'max_tokens'; if effort is provided we attempt to pass provider fields,
      and if rejected (HTTP 400), we retry without the effort fields (fallback to plain CoT if the prompt had it).
    Returns: dict with text, tokens, cost, and some echo of parameters.
    """
    cfg = MODEL_REGISTRY[model_key]
    provider = cfg["provider"]
    model = cfg["model"]

    # We always keep temperature at 0 for core experiments; SC-5 will override later.
    temperature = float(temperature)
    top_p = float(top_p)

    last_err = None
    for attempt in range(retries + 1):
        try:
            if provider == "openai":
                # Responses API (GPT‑5)
                kwargs = dict(
                    model=model,
                    input=prompt,
                    max_output_tokens=int(max_output_tokens),
                )
                if cfg.get("supports_reasoning_effort") and effort:
                    kwargs["reasoning"] = {"effort": effort}

                resp = openai_client.responses.create(**kwargs)
                text = _extract_text_openai(resp)
                pt, ct = _extract_usage_openai(resp)
                cost = estimate_cost_usd(pt, ct, cfg["price_in"], cfg["price_out"])
                return {
                    "provider": provider,
                    "model_key": model_key,
                    "model": model,
                    "mode": mode,
                    "text": text,
                    "prompt_tokens": pt,
                    "completion_tokens": ct,
                    "total_tokens": pt + ct,
                    "est_cost_usd": cost,
                    "applied_effort": effort if cfg.get("supports_reasoning_effort") else None,
                    "used_api": "responses"
                }

            else:
                # DeepInfra (OpenAI-compatible Chat Completions)
                params = dict(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                    top_p=top_p,
                    max_tokens=int(max_output_tokens),
                )

                # Try passing provider-side effort knobs if supported
                passed_effort_fields = False
                if cfg.get("supports_reasoning_effort") and effort:
                    # Different providers accept different keys; try both
                    params["reasoning"] = {"effort": effort}
                    params["reasoning_effort"] = effort
                    passed_effort_fields = True

                resp = deepinfra_client.chat.completions.create(**params)
                text = _extract_text_chat(resp)
                pt, ct = _extract_usage_chat(resp)
                cost = estimate_cost_usd(pt, ct, cfg["price_in"], cfg["price_out"])
                return {
                    "provider": provider,
                    "model_key": model_key,
                    "model": model,
                    "mode": mode,
                    "text": text,
                    "prompt_tokens": pt,
                    "completion_tokens": ct,
                    "total_tokens": pt + ct,
                    "est_cost_usd": cost,
                    "applied_effort": effort if passed_effort_fields else None,
                    "used_api": "chat.completions"
                }

        except Exception as e:
            last_err = e
            # If DeepInfra rejected the 'reasoning' fields, retry without them
            if provider != "openai" and attempt < retries:
                time.sleep(retry_backoff ** attempt)
                try:
                    params.pop("reasoning", None)
                    params.pop("reasoning_effort", None)
                except Exception:
                    pass
                continue
            # If OpenAI transient error, backoff and retry
            if provider == "openai" and attempt < retries:
                time.sleep(retry_backoff ** attempt)
                continue
            break

    # If we got here, all attempts failed
    raise RuntimeError(f"Model call failed for {model_key} / mode={mode}: {last_err}")


# -------------------------- Convenience wrappers -----------------------------

def build_direct_or_reasoning_prompt(ex: Dict[str,Any], model_key: str, mode: str) -> Tuple[str, Optional[str], int]:
    """
    Returns (prompt, effort, max_output_tokens) for MVP Direct/Reasoning.
    - For GLM‑Air & GPT‑5, 'reasoning' mode uses provider effort (text unchanged).
    - For the other models, 'reasoning' uses CoT prefix.
    """
    cfg = MODEL_REGISTRY[model_key]
    if mode == "direct":
        p = build_prompt(ex, "direct")
        eff = "minimal" if (cfg["provider"] == "openai" and cfg.get("supports_reasoning_effort")) else None
        cap = DIRECT_MAX if cfg.get("uses_max_output_tokens", False) else DIRECT_MAX
        return p, eff, cap

    # mode == "reasoning"
    if cfg.get("supports_reasoning_effort"):
        p = build_prompt(ex, "reasoning", use_cot_prefix=False)  # text same as 'direct'
        eff = "medium"
    else:
        p = build_prompt(ex, "reasoning", use_cot_prefix=True)
        eff = None
    cap = REASON_MAX
    return p, eff, cap


def parse_int_in_range(text: str, lo: int, hi: int) -> Optional[int]:
    if not text: return None
    m = re.search(r"(-?\d+)", text)
    if not m: return None
    val = int(m.group(1))
    if val < lo or val > hi: return None
    return val

def parse_budget_choice(text: str, allowed: List[int]) -> Optional[int]:
    if not text: return None
    m = re.search(r"(\d+)", text)
    if not m: return None
    val = int(m.group(1))
    return val if val in allowed else None


print("✅ STEP 3 ready: prompts, parsers, and unified callers are defined.")


✅ STEP 3 ready: prompts, parsers, and unified callers are defined.


In [8]:
# STEP 4 — Execution loops (MVP, DCAB, AV-vs-TA) with full persistence
# Requires:
#  - From Step 1: RUN_DIR, MODEL_REGISTRY, BUDGETS, DIRECT_MAX, REASON_MAX
#  - From Step 2: ITEMS (list of examples)
#  - From Step 3: build_prompt, build_direct_or_reasoning_prompt, call_model,
#                  extract_label, parse_int_in_range, parse_budget_choice, map_budget_to_effort

import os, json, pathlib, time
from collections import defaultdict
import pandas as pd
from tqdm import tqdm

# -------- Controls --------
RUN_MVP  = True
RUN_DCAB = True
RUN_AVTA = True

# Soft spend guard (approximate; we compute from token usage each call)
SOFT_SPEND_LIMIT_USD = 3.00  # adjust if you want more headroom
HALT_ON_LIMIT = True         # stop new calls when limit exceeded

# Whether to include full 'raw' model text in saved CSV/JSONL (for anonymization toggle)
EXPORT_RATIONALES = True

# -------- Utilities to save rows per (model, mode) --------
def _write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def _save_model_mode_rows(model_key, mode_tag, rows):
    base = RUN_DIR / "preds" / f"{model_key}__{mode_tag}"
    df = pd.DataFrame(rows)
    _write_jsonl(base.with_suffix(".jsonl"), rows)
    df.to_csv(base.with_suffix(".csv"), index=False)
    return df

def _empty_row_for(ex, model_key, provider, model, mode, text="", pt=0, ct=0, cost=0.0):
    return {
        "model_key": model_key, "provider": provider, "model": model,
        "mode": mode,
        "dataset": ex["dataset"], "depth": ex["depth"], "id": ex["id"],
        "gold": ex.get("answer", ""), "pred": "",
        "raw": (text if EXPORT_RATIONALES else None),
        "prompt_tokens": pt, "completion_tokens": ct, "est_cost_usd": round(cost, 6),
        "d_hat": None, "b_hat": None, "budget": None, "changed_by_verify": None
    }

def _finalize_row(row, pred):
    row["pred"] = pred if pred is not None else "UNK"
    return row

# -------- Spend tracker --------
total_spend = 0.0
spend_by_model = defaultdict(float)

def _accumulate_spend(model_key, cost):
    global total_spend
    total_spend += float(cost or 0.0)
    spend_by_model[model_key] += float(cost or 0.0)

def _maybe_halt():
    if HALT_ON_LIMIT and total_spend >= SOFT_SPEND_LIMIT_USD:
        print(f"\n⛔️ Soft spend limit reached (${total_spend:.3f} ≥ ${SOFT_SPEND_LIMIT_USD:.2f}). Halting new requests.")
        return True
    return False

# -------- Helper: run a single call & build a row --------
def _run_one(ex, model_key, mode, prompt, cap_tokens, effort=None):
    cfg = MODEL_REGISTRY[model_key]
    # Perform the API call
    out = call_model(model_key, prompt, mode=mode, max_output_tokens=cap_tokens, effort=effort)
    # Prepare row
    row = _empty_row_for(ex, model_key, out["provider"], out["model"], mode,
                         text=out["text"], pt=out["prompt_tokens"], ct=out["completion_tokens"], cost=out["est_cost_usd"])
    _accumulate_spend(model_key, out["est_cost_usd"])
    return out, row

# =============================== 4A) MVP =====================================
ALL_ROWS = []  # collect everything to concatenate at the end

if RUN_MVP:
    print("▶️ Running MVP (Direct vs Reasoning) ...")
    for model_key in MODEL_REGISTRY.keys():
        # Two modes per model
        for mode in ["direct", "reasoning"]:
            mode_rows = []
            if _maybe_halt(): break
            for ex in tqdm(ITEMS, desc=f"{model_key}:{mode}", leave=False):
                if _maybe_halt(): break
                prompt, eff, cap = build_direct_or_reasoning_prompt(ex, model_key, mode)
                try:
                    out, row = _run_one(ex, model_key, mode, prompt, cap_tokens=cap, effort=eff)
                    pred = extract_label(out["text"], ex["choices"]) or "UNK"
                    row = _finalize_row(row, pred)
                    mode_rows.append(row)
                    ALL_ROWS.append(row)
                except Exception as e:
                    # Log a failed row with empty pred (won't count in accuracy later)
                    row = _empty_row_for(ex, model_key, MODEL_REGISTRY[model_key]["provider"], MODEL_REGISTRY[model_key]["model"], mode)
                    row["raw"] = f"<ERROR: {e}>"
                    mode_rows.append(row)
                    ALL_ROWS.append(row)

            # Save per model/mode
            _save_model_mode_rows(model_key, mode, mode_rows)

# ============================ 4B) DCAB Protocol ===============================
if RUN_DCAB and not _maybe_halt():
    print("▶️ Running DCAB (Depth‑Calibration & Adaptive Budgeting) ...")
    for model_key in MODEL_REGISTRY.keys():
        mode_rows_meta = []   # optional: keep depth/budget predictions as separate logs
        mode_rows_final = []  # scored rows (dcab)
        if _maybe_halt(): break
        for ex in tqdm(ITEMS, desc=f"{model_key}:dcab", leave=False):
            if _maybe_halt(): break
            # ---- (1) Depth estimate ----
            p1 = build_prompt(ex, "dcab_depth")
            try:
                out1, row_depth = _run_one(ex, model_key, "dcab_depth", p1, cap_tokens=16, effort=None)
                d_hat = parse_int_in_range(out1["text"], 1, 6)  # allow 1..6
            except Exception as e:
                out1, row_depth = None, _empty_row_for(ex, model_key, MODEL_REGISTRY[model_key]["provider"], MODEL_REGISTRY[model_key]["model"], "dcab_depth")
                row_depth["raw"] = f"<ERROR: {e}>"
                d_hat = None
            row_depth["d_hat"] = d_hat
            mode_rows_meta.append(row_depth)
            ALL_ROWS.append(row_depth)

            # default depth if parse failed
            if d_hat is None:
                d_hat = 2 if ex["dataset"] == "clutrr" else 1

            # ---- (2) Budget proposal ----
            p2 = build_prompt(ex, "dcab_budget", d_hat=d_hat, budgets=BUDGETS)
            try:
                out2, row_budget = _run_one(ex, model_key, "dcab_budget", p2, cap_tokens=12, effort=None)
                b_hat = parse_budget_choice(out2["text"], BUDGETS)
            except Exception as e:
                out2, row_budget = None, _empty_row_for(ex, model_key, MODEL_REGISTRY[model_key]["provider"], MODEL_REGISTRY[model_key]["model"], "dcab_budget")
                row_budget["raw"] = f"<ERROR: {e}>"
                b_hat = None
            if b_hat is None:
                b_hat = 128
            row_budget["d_hat"] = d_hat
            row_budget["b_hat"] = b_hat
            mode_rows_meta.append(row_budget)
            ALL_ROWS.append(row_budget)

            # ---- (3) Solve under b_hat ----
            # Use provider effort mapping when available, else CoT with token cap
            if MODEL_REGISTRY[model_key].get("supports_reasoning_effort"):
                eff = map_budget_to_effort(b_hat)
                p3 = build_prompt(ex, "reasoning", use_cot_prefix=False)
            else:
                eff = None
                p3 = build_prompt(ex, "reasoning", use_cot_prefix=True)
            try:
                out3, row_final = _run_one(ex, model_key, "dcab", p3, cap_tokens=b_hat, effort=eff)
                pred = extract_label(out3["text"], ex["choices"]) or "UNK"
            except Exception as e:
                row_final = _empty_row_for(ex, model_key, MODEL_REGISTRY[model_key]["provider"], MODEL_REGISTRY[model_key]["model"], "dcab")
                row_final["raw"] = f"<ERROR: {e}>"
                pred = "UNK"
            row_final["d_hat"] = d_hat
            row_final["b_hat"] = b_hat
            row_final = _finalize_row(row_final, pred)
            mode_rows_final.append(row_final)
            ALL_ROWS.append(row_final)

        # Save meta (depth + budget) and final scored rows
        _save_model_mode_rows(model_key, "dcab_meta", mode_rows_meta)
        _save_model_mode_rows(model_key, "dcab", mode_rows_final)

# ========================== 4C) AV vs TA (matched) ============================
if RUN_AVTA and not _maybe_halt():
    print("▶️ Running AV‑vs‑TA (matched budgets) ...")
    for model_key in MODEL_REGISTRY.keys():
        if _maybe_halt(): break
        for B in BUDGETS:
            rows_ta = []
            rows_av = []
            # ---- Think‑then‑Answer (TA) ----
            for ex in tqdm(ITEMS, desc=f"{model_key}:TA@{B}", leave=False):
                if _maybe_halt(): break
                # Build 'reasoning' prompt (provider effort if supported)
                if MODEL_REGISTRY[model_key].get("supports_reasoning_effort"):
                    eff_ta = map_budget_to_effort(B)
                    p_ta = build_prompt(ex, "reasoning", use_cot_prefix=False)
                else:
                    eff_ta = None
                    p_ta = build_prompt(ex, "reasoning", use_cot_prefix=True)
                try:
                    out_ta, row_ta = _run_one(ex, model_key, f"ta_{B}", p_ta, cap_tokens=B, effort=eff_ta)
                    pred_ta = extract_label(out_ta["text"], ex["choices"]) or "UNK"
                except Exception as e:
                    row_ta = _empty_row_for(ex, model_key, MODEL_REGISTRY[model_key]["provider"], MODEL_REGISTRY[model_key]["model"], f"ta_{B}")
                    row_ta["raw"] = f"<ERROR: {e}>"
                    pred_ta = "UNK"
                row_ta["budget"] = B
                row_ta = _finalize_row(row_ta, pred_ta)
                rows_ta.append(row_ta)
                ALL_ROWS.append(row_ta)

            _save_model_mode_rows(model_key, f"ta_{B}", rows_ta)

            # ---- Answer‑then‑Verify (AV) ----
            for ex in tqdm(ITEMS, desc=f"{model_key}:AV@{B}", leave=False):
                if _maybe_halt(): break
                # 1) Short answer (Direct@32)
                p_ans = build_prompt(ex, "direct")
                try:
                    out_a, row_a = _run_one(ex, model_key, f"av_answer@{B}", p_ans, cap_tokens=32, effort=None)
                    pred0 = extract_label(out_a["text"], ex["choices"]) or "UNK"
                except Exception as e:
                    row_a = _empty_row_for(ex, model_key, MODEL_REGISTRY[model_key]["provider"], MODEL_REGISTRY[model_key]["model"], f"av_answer@{B}")
                    row_a["raw"] = f"<ERROR: {e}>"
                    pred0 = "UNK"
                row_a["budget"] = 32
                rows_av.append(row_a)
                ALL_ROWS.append(row_a)

                # 2) Verify step (B-32)
                rem = max(B - 32, 0)
                p_ver = build_prompt(ex, "verify", proposed=pred0)
                try:
                    out_v, row_v = _run_one(ex, model_key, f"av_verify@{B}", p_ver, cap_tokens=rem, effort=None)
                    pred_v = extract_label(out_v["text"], ex["choices"]) or pred0
                except Exception as e:
                    row_v = _empty_row_for(ex, model_key, MODEL_REGISTRY[model_key]["provider"], MODEL_REGISTRY[model_key]["model"], f"av_verify@{B}")
                    row_v["raw"] = f"<ERROR: {e}>"
                    pred_v = pred0

                # Final AV row (scored)
                row_final = _empty_row_for(ex, model_key, row_v["provider"], row_v["model"], f"av_{B}",
                                           text=(row_v["raw"] if EXPORT_RATIONALES else None),
                                           pt=row_v["prompt_tokens"], ct=row_v["completion_tokens"], cost=row_v["est_cost_usd"])
                row_final["budget"] = B
                row_final["changed_by_verify"] = (pred_v != pred0)
                row_final = _finalize_row(row_final, pred_v)

                rows_av.append(row_v)       # keep verify log
                rows_av.append(row_final)   # keep final scored AV
                ALL_ROWS.append(row_v)
                ALL_ROWS.append(row_final)

            # Save AV logs per budget (answer, verify, and final rows together)
            _save_model_mode_rows(model_key, f"av@{B}", rows_av)

# ============================== Save everything ===============================
all_df = pd.DataFrame(ALL_ROWS)
all_path = RUN_DIR / "all_predictions.csv"
all_df.to_csv(all_path, index=False)

# Spend summary
spend_rows = [{"model_key": k, "est_cost_usd": round(v, 4)} for k, v in sorted(spend_by_model.items(), key=lambda kv: kv[0])]
spend_df = pd.DataFrame(spend_rows).sort_values("est_cost_usd", ascending=False)
spend_df.to_csv(RUN_DIR / "tables" / "spend_preview.csv", index=False)

print("\n✅ STEP 4 complete.")
print(f"Total rows saved: {len(all_df)}")
print(f"All predictions → {all_path}")
print("\nEstimated spend by model:")
display(spend_df)


▶️ Running MVP (Direct vs Reasoning) ...


▶️ Running DCAB (Depth‑Calibration & Adaptive Budgeting) ...


▶️ Running AV‑vs‑TA (matched budgets) ...



✅ STEP 4 complete.
Total rows saved: 8500
All predictions → /content/runs/20250814-071739/all_predictions.csv

Estimated spend by model:


,model_key,est_cost_usd
2,gpt5,1.3530
1,glm45_air,0.0876
0,deepseek_r1d_qwen32b,0.0382
4,qwen25_7b,0.0103
3,llama31_8b,0.0078


In [10]:
# STEP 5 — Aggregations, metrics, and plots
# Inputs: runs/<ts>/all_predictions.csv (from STEP 4)
# Outputs (CSV): accuracy_by_depth.csv, delta_reasoning.csv, collapse_points.csv,
#                dcab_calibration.csv, dcab_budget_efficiency.csv, dcab_collapse_shift.csv,
#                av_vs_ta_accuracy.csv, av_refutation_rate.csv, spend_summary.csv
# Outputs (PNG): plots/acc_{model}_{dataset}.png, plots/delta_{model}_{dataset}.png

import os, re, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

tables_dir = RUN_DIR / "tables"
plots_dir  = RUN_DIR / "plots"
tables_dir.mkdir(exist_ok=True, parents=True)
plots_dir.mkdir(exist_ok=True, parents=True)

# --------------------------- Load & prep ---------------------------
all_path = RUN_DIR / "all_predictions.csv"
assert all_path.exists(), f"Missing {all_path}. Run STEP 4 first."

df = pd.read_csv(all_path)
# Normalize dtypes
for col in ["prompt_tokens","completion_tokens","est_cost_usd","depth","budget","d_hat","b_hat"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df["total_tokens"] = (df["prompt_tokens"].fillna(0) + df["completion_tokens"].fillna(0)).astype(float)

# Correctness (UNK or empty counts as wrong)
def _norm(s):
    return str(s).strip().lower() if isinstance(s, str) else str(s).lower()
df["correct"] = (df["gold"].map(_norm) == df["pred"].map(_norm)).astype(int)

# --------------------------- Helper funcs ---------------------------
def accuracy_table(sub):
    """Return accuracy grouped by model_key, mode, dataset, depth."""
    return (sub.groupby(["model_key","mode","dataset","depth"])["correct"]
              .mean()
              .reset_index())

def pivot_accuracy(acc_df):
    """Pivot to Table-2 style percentages."""
    pv = (acc_df.pivot_table(index=["model_key","mode"], columns=["dataset","depth"], values="correct")
                 .sort_index())
    return (pv * 100).round(1)

def collapse_point_for(sub):
    """Smallest depth with accuracy < 0.5; returns None if never collapses."""
    depths = sorted(sub["depth"].unique())
    for d in depths:
        acc_d = sub.loc[sub["depth"]==d, "correct"].mean()
        if pd.notnull(acc_d) and acc_d < 0.5:
            return int(d)
    return None

def parse_budget_from_mode(mode):
    # modes like "ta_128", "av_64", "av_answer@128", "av_verify@128"
    m = re.search(r"(\d+)", str(mode))
    return int(m.group(1)) if m else None

# --------------------- MVP: accuracy, delta, collapse ------------------------
mvp = df[df["mode"].isin(["direct","reasoning"])].copy()
acc = accuracy_table(mvp)
table2 = pivot_accuracy(acc)
table2.to_csv(tables_dir / "accuracy_by_depth.csv")

# ΔReasoning = Acc(reasoning) − Acc(direct)
direct = (acc[acc["mode"]=="direct"]
          .set_index(["model_key","dataset","depth"])["correct"])
reason = (acc[acc["mode"]=="reasoning"]
          .set_index(["model_key","dataset","depth"])["correct"])
delta = (reason - direct).reset_index(name="delta").sort_values(["model_key","dataset","depth"])
delta.to_csv(tables_dir / "delta_reasoning.csv", index=False)

# Collapse points for direct & reasoning (per dataset)
rows_cp = []
for (mk, mode, dsname), sub in acc.groupby(["model_key","mode","dataset"]):
    rows_cp.append({
        "model_key": mk, "mode": mode, "dataset": dsname,
        "collapse_point": collapse_point_for(sub)
    })
collapse_df = pd.DataFrame(rows_cp)
collapse_df.to_csv(tables_dir / "collapse_points.csv", index=False)

# --------------------- DCAB: calibration (Spearman, ECE) ---------------------
dc_depth = df[df["mode"]=="dcab_depth"].copy()
cal_rows = []

def compute_ece_ord(d_true, d_pred, min_d, max_d):
    """
    Ordinal ECE: weight by P(D=d), compare mean predicted depth in each true bin d to d.
    Normalize by range so ECE ∈ [0,1].
    """
    d_true = np.asarray(d_true, dtype=float)
    d_pred = np.asarray(d_pred, dtype=float)
    mask = ~np.isnan(d_true) & ~np.isnan(d_pred)
    d_true, d_pred = d_true[mask], d_pred[mask]
    if len(d_true) == 0:
        return np.nan
    ece = 0.0
    rng = max(1.0, float(max_d - min_d))
    vals, counts = np.unique(d_true, return_counts=True)
    n = len(d_true)
    for v, c in zip(vals, counts):
        m = d_pred[d_true == v].mean()
        ece += (c / n) * (abs(m - v) / rng)
    return float(ece)

for mk, g in dc_depth.groupby("model_key"):
    # Overall across datasets
    g2 = g.dropna(subset=["d_hat","depth"])
    rho_overall = g2["d_hat"].corr(g2["depth"], method="spearman") if len(g2)>=2 else np.nan

    # Compute ECE using observed min/max across both datasets
    if len(g2) > 0:
        dmin, dmax = int(g2["depth"].min()), int(g2["depth"].max())
        ece_overall = compute_ece_ord(g2["depth"], g2["d_hat"], dmin, dmax)
    else:
        ece_overall = np.nan

    cal_rows.append({"model_key": mk, "dataset": "overall", "n": int(len(g2)),
                     "spearman_rho": round(float(rho_overall), 4) if pd.notnull(rho_overall) else np.nan,
                     "ece": round(float(ece_overall), 4) if pd.notnull(ece_overall) else np.nan})

    # Per dataset
    for dsname, gds in g.groupby("dataset"):
        gds2 = gds.dropna(subset=["d_hat","depth"])
        rho = gds2["d_hat"].corr(gds2["depth"], method="spearman") if len(gds2)>=2 else np.nan
        if len(gds2) > 0:
            dmin, dmax = int(gds2["depth"].min()), int(gds2["depth"].max())
            ece = compute_ece_ord(gds2["depth"], gds2["d_hat"], dmin, dmax)
        else:
            ece = np.nan
        cal_rows.append({"model_key": mk, "dataset": dsname, "n": int(len(gds2)),
                         "spearman_rho": round(float(rho), 4) if pd.notnull(rho) else np.nan,
                         "ece": round(float(ece), 4) if pd.notnull(ece) else np.nan})

dcab_cal = pd.DataFrame(cal_rows).sort_values(["model_key","dataset"])
dcab_cal.to_csv(tables_dir / "dcab_calibration.csv", index=False)

# --------------------- Budget efficiency / MB5 ---------------------
# Average tokens for Direct (baseline)
direct_stats = (df[df["mode"]=="direct"]
                .groupby(["model_key","dataset","depth"])
                .agg(acc=("correct","mean"),
                     avg_tokens=("total_tokens","mean"))
                .reset_index()
                .rename(columns={"acc":"acc_direct","avg_tokens":"tok_direct"}))

# Reasoning baseline (single cap REASON_MAX)
reason_stats = (df[df["mode"]=="reasoning"]
                .groupby(["model_key","dataset","depth"])
                .agg(acc=("correct","mean"),
                     avg_tokens=("total_tokens","mean"))
                .reset_index()
                .rename(columns={"acc":"acc_reason","avg_tokens":"tok_reason"}))

# DCAB: combine depth+budget+solve tokens per item
depth_tok = (df[df["mode"]=="dcab_depth"]
             .groupby(["model_key","dataset","depth","id"])
             .agg(tok=("total_tokens","sum"))
             .reset_index()
             .rename(columns={"tok":"tok_dc_depth"}))
budget_tok = (df[df["mode"]=="dcab_budget"]
             .groupby(["model_key","dataset","depth","id"])
             .agg(tok=("total_tokens","sum"))
             .reset_index()
             .rename(columns={"tok":"tok_dc_budget"}))
solve_dc = df[df["mode"]=="dcab"][["model_key","dataset","depth","id","correct","total_tokens"]].copy()
solve_dc = solve_dc.rename(columns={"total_tokens":"tok_dc_solve"})
dcab_merge = solve_dc.merge(depth_tok, on=["model_key","dataset","depth","id"], how="left") \
                     .merge(budget_tok, on=["model_key","dataset","depth","id"], how="left")
dcab_merge["tok_dc_total"] = dcab_merge[["tok_dc_solve","tok_dc_depth","tok_dc_budget"]].fillna(0).sum(axis=1)
dcab_stats = (dcab_merge.groupby(["model_key","dataset","depth"])
              .agg(acc_dcab=("correct","mean"),
                   tok_dcab=("tok_dc_total","mean"))
              .reset_index())

# TA budgets
def agg_ta(B):
    sub = df[df["mode"]==f"ta_{B}"]
    return (sub.groupby(["model_key","dataset","depth"])
            .agg(**{f"acc_ta_{B}":("correct","mean"),
                    f"tok_ta_{B}":("total_tokens","mean")})
            .reset_index())

ta_stats = None
for B in BUDGETS:
    cur = agg_ta(B)
    ta_stats = cur if ta_stats is None else ta_stats.merge(cur, on=["model_key","dataset","depth"], how="outer")

# AV budgets (need to sum answer+verify tokens, use final 'av_B' for correctness)
def agg_av(B):
    ans = df[df["mode"]==f"av_answer@{B}"][["model_key","dataset","depth","id","total_tokens","correct"]]
    ver = df[df["mode"]==f"av_verify@{B}"][["model_key","dataset","depth","id","total_tokens","correct"]]
    fin = df[df["mode"]==f"av_{B}"][["model_key","dataset","depth","id","correct","changed_by_verify"]]
    merged = fin.merge(ans, on=["model_key","dataset","depth","id"], how="left", suffixes=("","_ans")) \
                .merge(ver, on=["model_key","dataset","depth","id"], how="left", suffixes=("","_ver"))
    merged["tok_av_total"] = merged["total_tokens_ans"].fillna(0) + merged["total_tokens_ver"].fillna(0)
    # group metrics
    acc = merged.groupby(["model_key","dataset","depth"])["correct"].mean().rename(f"acc_av_{B}")
    tok = merged.groupby(["model_key","dataset","depth"])["tok_av_total"].mean().rename(f"tok_av_{B}")
    refute_num = ((merged["correct_ans"]==0) & (merged["correct"]==1)).groupby(
        [merged["model_key"], merged["dataset"], merged["depth"]]).sum().rename(f"refute_count_{B}")
    wrong_ans = (merged["correct_ans"]==0).groupby(
        [merged["model_key"], merged["dataset"], merged["depth"]]).sum().rename(f"wrong_initial_count_{B}")
    out = pd.concat([acc, tok, refute_num, wrong_ans], axis=1).reset_index()
    return out

# Compute AV tables; note: 'correct_ans' column created above via renaming may not exist—handle explicitly
# --- HOTFIX: robust AV aggregator for Step 5 (replaces your agg_av_safe) ---
def agg_av_safe(B, df_in=None):
    """
    Returns:
      av_tbl: per (model_key, dataset, depth) -> acc_av, tok_av, wrong_initial_count, refute_count, budget
      merged: per-item merged table to optionally inspect later
    """
    dfin = df if df_in is None else df_in

    ans = dfin[dfin["mode"]==f"av_answer@{B}"][["model_key","dataset","depth","id","total_tokens","correct"]] \
            .rename(columns={"total_tokens":"tok_ans","correct":"correct_ans"})
    ver = dfin[dfin["mode"]==f"av_verify@{B}"][["model_key","dataset","depth","id","total_tokens","correct"]] \
            .rename(columns={"total_tokens":"tok_ver","correct":"correct_ver"})
    fin = dfin[dfin["mode"]==f"av_{B}"][["model_key","dataset","depth","id","correct"]] \
            .rename(columns={"correct":"correct_final"})

    merged = fin.merge(ans, on=["model_key","dataset","depth","id"], how="left") \
                .merge(ver, on=["model_key","dataset","depth","id"], how="left")
    merged["tok_av_total"] = merged["tok_ans"].fillna(0) + merged["tok_ver"].fillna(0)

    keys = ["model_key","dataset","depth"]

    # Core stats (column-wise aggregations)
    acc_tok = (merged.groupby(keys)
               .agg(acc_av=("correct_final","mean"),
                    tok_av=("tok_av_total","mean"))
               .reset_index())

    # Cross-column counts computed separately then merged
    wrong_initial = ((merged["correct_ans"]==0)
                     .groupby([merged[k] for k in keys]).sum()
                     .reset_index(name="wrong_initial_count"))
    flips = (((merged["correct_ans"]==0) & (merged["correct_final"]==1))
             .groupby([merged[k] for k in keys]).sum()
             .reset_index(name="refute_count"))

    out = (acc_tok.merge(wrong_initial, on=keys, how="left")
                 .merge(flips, on=keys, how="left"))
    out["wrong_initial_count"] = out["wrong_initial_count"].fillna(0).astype(int)
    out["refute_count"] = out["refute_count"].fillna(0).astype(int)
    out["budget"] = B
    return out, merged


av_stats_list = []
av_merged_store = []
for B in BUDGETS:
    av_tbl, merged_tbl = agg_av_safe(B)
    av_stats_list.append(av_tbl.rename(columns={"acc_av":"acc_av_"+str(B), "tok_av":"tok_av_"+str(B)}))
    av_merged_store.append((B, merged_tbl))

# Combine AV stats across budgets
av_stats = None
for tbl in av_stats_list:
    av_stats = tbl if av_stats is None else av_stats.merge(tbl, on=["model_key","dataset","depth","budget"], how="outer")
# Also produce refutation rate table
ref_rows = []
for B, merged in av_merged_store:
    grp = merged.groupby(["model_key","dataset","depth"])
    wrong = grp.apply(lambda g: (g["correct_ans"]==0).sum()).rename("wrong_initial")
    flips = grp.apply(lambda g: ((g["correct_ans"]==0) & (g["correct_final"]==1)).sum()).rename("flips_to_right")
    ref = pd.concat([wrong, flips], axis=1).reset_index()
    ref["budget"] = B
    ref["refute_rate"] = np.where(ref["wrong_initial"]>0, ref["flips_to_right"]/ref["wrong_initial"], np.nan)
    ref_rows.append(ref)
ref_tbl = pd.concat(ref_rows, axis=0, ignore_index=True)
ref_tbl.to_csv(tables_dir / "av_refutation_rate.csv", index=False)

# Merge all baselines for MB5 computation
merged_all = direct_stats \
    .merge(reason_stats, on=["model_key","dataset","depth"], how="outer") \
    .merge(dcab_stats,   on=["model_key","dataset","depth"], how="outer")

# Attach TA per-budget
for B in BUDGETS:
    merged_all = merged_all.merge(
        ta_stats[["model_key","dataset","depth", f"acc_ta_{B}", f"tok_ta_{B}"]],
        on=["model_key","dataset","depth"], how="outer"
    )
# Attach AV per-budget
for B in BUDGETS:
    # av_stats has rows per budget; pivot them into columns for merge
    av_cols = av_stats[av_stats["budget"]==B][["model_key","dataset","depth", "acc_av_"+str(B), "tok_av_"+str(B)]]
    merged_all = merged_all.merge(av_cols, on=["model_key","dataset","depth"], how="outer")

# Compute MB5 per (model_key, dataset, depth)
mb5_rows = []
for (mk, ds, d), g in merged_all.groupby(["model_key","dataset","depth"]):
    direct_acc = g["acc_direct"].iloc[0] if "acc_direct" in g else np.nan
    direct_tok = g["tok_direct"].iloc[0] if "tok_direct" in g else np.nan
    target = None if pd.isna(direct_acc) else direct_acc + 0.05

    candidates = []
    # Reasoning single point
    if "acc_reason" in g and pd.notna(g["acc_reason"].iloc[0]) and pd.notna(direct_tok):
        candidates.append(("reasoning", g["tok_reason"].iloc[0] - direct_tok, g["acc_reason"].iloc[0]))
    # DCAB
    if "acc_dcab" in g and pd.notna(g["acc_dcab"].iloc[0]) and pd.notna(direct_tok):
        candidates.append(("dcab", g["tok_dcab"].iloc[0] - direct_tok, g["acc_dcab"].iloc[0]))
    # TA budgets
    for B in BUDGETS:
        acc_col, tok_col = f"acc_ta_{B}", f"tok_ta_{B}"
        if acc_col in g and tok_col in g and pd.notna(g[acc_col].iloc[0]) and pd.notna(direct_tok):
            candidates.append((f"ta_{B}", g[tok_col].iloc[0] - direct_tok, g[acc_col].iloc[0]))
    # AV budgets
    for B in BUDGETS:
        acc_col, tok_col = f"acc_av_{B}", f"tok_av_{B}"
        if acc_col in g and tok_col in g and pd.notna(g[acc_col].iloc[0]) and pd.notna(direct_tok):
            candidates.append((f"av_{B}", g[tok_col].iloc[0] - direct_tok, g[acc_col].iloc[0]))

    # Select minimal extra tokens that achieves >= target
    best_method, best_extra, best_acc = None, np.nan, np.nan
    if target is not None:
        feasible = [(m, extra, accv) for (m, extra, accv) in candidates if pd.notna(accv) and accv >= target and pd.notna(extra)]
        if len(feasible) > 0:
            m, extra, accv = sorted(feasible, key=lambda t: (t[1], -t[2]))[0]
            best_method, best_extra, best_acc = m, float(extra), float(accv)

    mb5_rows.append({
        "model_key": mk, "dataset": ds, "depth": d,
        "acc_direct": round(float(direct_acc),4) if pd.notnull(direct_acc) else np.nan,
        "tok_direct": round(float(direct_tok),1) if pd.notnull(direct_tok) else np.nan,
        "MB5_method": best_method,
        "MB5_tokens": round(best_extra,1) if pd.notnull(best_extra) else np.nan,
        "MB5_acc": round(best_acc,4) if pd.notnull(best_acc) else np.nan
    })

mb5_tbl = pd.DataFrame(mb5_rows).sort_values(["model_key","dataset","depth"])
mb5_tbl.to_csv(tables_dir / "dcab_budget_efficiency.csv", index=False)

# --------------------- DCAB collapse shift vs best fixed budget --------------
# Best fixed = TA_B* where mean accuracy across depths is maximal (tie -> smaller B)
best_fixed_rows = []
for (mk, ds), g in ta_stats.groupby(["model_key","dataset"]):
    best_B, best_mean = None, -1
    for B in BUDGETS:
        col = f"acc_ta_{B}"
        if col in g and g[col].notna().any():
            mean_acc = g[col].mean()
            if (mean_acc > best_mean) or (np.isclose(mean_acc, best_mean) and (best_B is None or B < best_B)):
                best_mean, best_B = mean_acc, B
    if best_B is not None:
        best_fixed_rows.append({"model_key": mk, "dataset": ds, "best_fixed_B": best_B, "best_fixed_mean_acc": best_mean})

best_fixed_df = pd.DataFrame(best_fixed_rows)

# Collapse for DCAB vs best fixed
def collapse_from_table(tbl, col_acc, mk, ds):
    sub = tbl[(tbl["model_key"]==mk) & (tbl["dataset"]==ds)]
    if sub.empty: return None
    # Need per-depth accuracies
    depths = sorted(sub["depth"].unique())
    for d in depths:
        ad = float(sub.loc[sub["depth"]==d, col_acc].mean())
        if ad < 0.5:
            return int(d)
    return None

shift_rows = []
for _, r in best_fixed_df.iterrows():
    mk, ds, B = r["model_key"], r["dataset"], int(r["best_fixed_B"])
    # Per-depth tables for TA_B and DCAB
    ta_tbl = ta_stats[["model_key","dataset","depth", f"acc_ta_{B}"]].rename(columns={f"acc_ta_{B}":"acc"})
    dc_tbl = dcab_stats.rename(columns={"acc_dcab":"acc"})[["model_key","dataset","depth","acc"]]
    # collapse points
    cp_fixed = collapse_from_table(ta_tbl, "acc", mk, ds)
    cp_dcab  = collapse_from_table(dc_tbl, "acc", mk, ds)
    shift_rows.append({"model_key": mk, "dataset": ds,
                       "best_fixed_B": B,
                       "collapse_d_fixed": cp_fixed,
                       "collapse_d_dcab": cp_dcab,
                       "shift": (None if (cp_fixed is None or cp_dcab is None) else (cp_dcab - cp_fixed))})
dcab_shift = pd.DataFrame(shift_rows)
dcab_shift.to_csv(tables_dir / "dcab_collapse_shift.csv", index=False)

# --------------------- AV vs TA accuracy (matched budgets) --------------------
av_vs_ta_rows = []
for B in BUDGETS:
    # TA
    ta_sub = ta_stats[["model_key","dataset","depth", f"acc_ta_{B}"]].rename(columns={f"acc_ta_{B}":"acc_ta"})
    # AV
    av_sub = av_stats[av_stats["budget"]==B][["model_key","dataset","depth","acc_av_"+str(B)]].rename(columns={f"acc_av_{B}":"acc_av"})
    merged = ta_sub.merge(av_sub, on=["model_key","dataset","depth"], how="outer")
    merged["budget"] = B
    av_vs_ta_rows.append(merged)
av_vs_ta_tbl = pd.concat(av_vs_ta_rows, axis=0, ignore_index=True)
av_vs_ta_tbl.to_csv(tables_dir / "av_vs_ta_accuracy.csv", index=False)

# --------------------- Spend summary (recompute precisely) --------------------
spend = (df.groupby(["model_key","mode"])
           .agg(total_prompt_tokens=("prompt_tokens","sum"),
                total_completion_tokens=("completion_tokens","sum"),
                est_cost_usd=("est_cost_usd","sum"))
           .reset_index())
spend["est_cost_usd"] = spend["est_cost_usd"].round(4)
spend.to_csv(tables_dir / "spend_summary.csv", index=False)

# --------------------- Plots: Accuracy & ΔReasoning --------------------------
def plot_acc_depth(model_key, dataset, savepath):
    sub = acc[(acc["model_key"]==model_key) & (acc["dataset"]==dataset)]
    if sub.empty:
        return
    depths = sorted(sub["depth"].unique())
    acc_d = [sub[(sub["mode"]=="direct") & (sub["depth"]==d)]["correct"].mean()*100 for d in depths]
    acc_r = [sub[(sub["mode"]=="reasoning") & (sub["depth"]==d)]["correct"].mean()*100 for d in depths]
    plt.figure()
    plt.plot(depths, acc_d, marker="o", label="Direct")
    plt.plot(depths, acc_r, marker="o", label="Reasoning")
    plt.xlabel("Depth")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Accuracy vs Depth — {model_key} on {dataset}")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.savefig(savepath, bbox_inches="tight")
    plt.close()

def plot_delta_depth(model_key, dataset, savepath):
    sub = delta[(delta["model_key"]==model_key) & (delta["dataset"]==dataset)].sort_values("depth")
    if sub.empty:
        return
    plt.figure()
    plt.plot(sub["depth"].tolist(), (sub["delta"]*100).tolist(), marker="o")
    plt.xlabel("Depth")
    plt.ylabel("ΔReasoning (pp)")
    plt.title(f"ΔReasoning vs Depth — {model_key} on {dataset}")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.savefig(savepath, bbox_inches="tight")
    plt.close()

for mk in sorted(df["model_key"].unique()):
    for ds in sorted(df["dataset"].unique()):
        plot_acc_depth(mk, ds, plots_dir / f"acc_{mk}_{ds}.png")
        plot_delta_depth(mk, ds, plots_dir / f"delta_{mk}_{ds}.png")

# --------------------- Persist “publication-ready” Table 2 --------------------
# Round to 1 decimal; fill missing with blank
table2_pub = table2.copy().fillna("")
table2_pub.to_csv(tables_dir / "table2_publication_ready.csv")

# --------------------- Console summary --------------------
print("✅ STEP 5 complete.")
print("Saved tables:")
for p in sorted(tables_dir.glob("*.csv")):
    print("  -", p.name)
print("\nSaved plots:")
for p in sorted(plots_dir.glob("*.png")):
    print("  -", p.name)

# Small previews
print("\nTable 2 (head):")
display(table2.head(10))
print("\nΔReasoning (head):")
display(delta.head(10))
print("\nDCAB calibration (head):")
display(dcab_cal.head(10))
print("\nMB5 (head):")
display(mb5_tbl.head(10))
print("\nAV vs TA accuracy (head):")
display(av_vs_ta_tbl.head(10))
print("\nRefutation rate (head):")
display(ref_tbl.head(10))


/tmp/ipython-input-1447318202.py:271: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wrong = grp.apply(lambda g: (g["correct_ans"]==0).sum()).rename("wrong_initial")
/tmp/ipython-input-1447318202.py:272: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  flips = grp.apply(lambda g: ((g["correct_ans"]==0) & (g["correct_final"]==1)).sum()).rename("flips_to_right")
/tmp/ipython-input-1447318202.py:271: DeprecationWa

✅ STEP 5 complete.
Saved tables:
  - accuracy_by_depth.csv
  - av_refutation_rate.csv
  - av_vs_ta_accuracy.csv
  - collapse_points.csv
  - dcab_budget_efficiency.csv
  - dcab_calibration.csv
  - dcab_collapse_shift.csv
  - delta_reasoning.csv
  - items_count_by_depth.csv
  - items_preview.csv
  - spend_preview.csv
  - spend_summary.csv
  - table2_publication_ready.csv

Saved plots:
  - acc_deepseek_r1d_qwen32b_clutrr.png
  - acc_deepseek_r1d_qwen32b_proofwriter.png
  - acc_glm45_air_clutrr.png
  - acc_glm45_air_proofwriter.png
  - acc_gpt5_clutrr.png
  - acc_gpt5_proofwriter.png
  - acc_llama31_8b_clutrr.png
  - acc_llama31_8b_proofwriter.png
  - acc_qwen25_7b_clutrr.png
  - acc_qwen25_7b_proofwriter.png
  - delta_deepseek_r1d_qwen32b_clutrr.png
  - delta_deepseek_r1d_qwen32b_proofwriter.png
  - delta_glm45_air_clutrr.png
  - delta_glm45_air_proofwriter.png
  - delta_gpt5_clutrr.png
  - delta_gpt5_proofwriter.png
  - delta_llama31_8b_clutrr.png
  - delta_llama31_8b_proofwriter.png
  -

dataset                        clutrr                         proofwriter  \
depth                               2     3     4     5     6           1   
model_key            mode                                                   
deepseek_r1d_qwen32b direct       0.0   0.0   0.0   0.0   0.0         0.0   
                     reasoning   10.0   0.0   0.0   0.0  10.0         0.0   
glm45_air            direct      10.0  10.0   0.0   0.0   0.0         0.0   
                     reasoning    0.0   0.0   0.0   0.0   0.0         0.0   
gpt5                 direct      10.0  10.0  10.0  10.0  10.0        50.0   
                     reasoning   20.0   0.0   0.0   0.0   0.0        20.0   
llama31_8b           direct      20.0  20.0  20.0  20.0  20.0        50.0   
                     reasoning   30.0  20.0  20.0  20.0  20.0        50.0   
qwen25_7b            direct      10.0   0.0   0.0  30.0   0.0        40.0   
                     reasoning   10.0   0.0  20.0   0.0   0.0        30.0   

dataset                                                 
depth                              2     3     4     5  
model_key            mode                               
deepseek_r1d_qwen32b direct      0.0   0.0   0.0   0.0  
                     reasoning   0.0   0.0   0.0   0.0  
glm45_air            direct      0.0   0.0   0.0   0.0  
                     reasoning   0.0   0.0   0.0   0.0  
gpt5                 direct     60.0  80.0  90.0  80.0  
                     reasoning  20.0  10.0   0.0   0.0  
llama31_8b           direct     20.0  50.0  40.0  60.0  
                     reasoning  20.0  50.0  50.0  50.0  
qwen25_7b            direct     30.0  40.0  30.0  60.0  
                     reasoning  20.0  40.0  40.0  50.0


ΔReasoning (head):


,model_key,dataset,depth,delta
0,deepseek_r1d_qwen32b,clutrr,2,0.1
1,deepseek_r1d_qwen32b,clutrr,3,0.0
2,deepseek_r1d_qwen32b,clutrr,4,0.0
3,deepseek_r1d_qwen32b,clutrr,5,0.0
4,deepseek_r1d_qwen32b,clutrr,6,0.1
5,deepseek_r1d_qwen32b,proofwriter,1,0.0
6,deepseek_r1d_qwen32b,proofwriter,2,0.0
7,deepseek_r1d_qwen32b,proofwriter,3,0.0
8,deepseek_r1d_qwen32b,proofwriter,4,0.0
9,deepseek_r1d_qwen32b,proofwriter,5,0.0


...

,model_key,dataset,depth,acc_direct,tok_direct,MB5_method,MB5_tokens,MB5_acc
0,deepseek_r1d_qwen32b,clutrr,2,0.0,151.8,ta_64,39.0,0.1
1,deepseek_r1d_qwen32b,clutrr,3,0.0,177.0,ta_256,216.0,0.1
2,deepseek_r1d_qwen32b,clutrr,4,0.0,183.7,None,NaN,NaN
3,deepseek_r1d_qwen32b,clutrr,5,0.0,196.7,None,NaN,NaN
4,deepseek_r1d_qwen32b,clutrr,6,0.0,219.3,ta_64,39.0,0.1
5,deepseek_r1d_qwen32b,proofwriter,1,0.0,188.6,av_64,243.6,0.1
6,deepseek_r1d_qwen32b,proofwriter,2,0.0,194.8,av_64,249.8,0.2
7,deepseek_r1d_qwen32b,proofwriter,3,0.0,197.5,av_64,252.5,0.1
8,deepseek_r1d_qwen32b,proofwriter,4,0.0,247.8,av_64,302.8,0.3
9,deepseek_r1d_qwen32b,proofwriter,5,0.0,224.9,av_64,279.9,0.2



AV vs TA accuracy (head):


,model_key,dataset,depth,acc_ta,acc_av,budget
0,deepseek_r1d_qwen32b,clutrr,2,0.1,0.0,64
1,deepseek_r1d_qwen32b,clutrr,3,0.0,0.0,64
2,deepseek_r1d_qwen32b,clutrr,4,0.0,0.0,64
3,deepseek_r1d_qwen32b,clutrr,5,0.0,0.0,64
4,deepseek_r1d_qwen32b,clutrr,6,0.1,0.0,64
5,deepseek_r1d_qwen32b,proofwriter,1,0.0,0.1,64
6,deepseek_r1d_qwen32b,proofwriter,2,0.0,0.2,64
7,deepseek_r1d_qwen32b,proofwriter,3,0.0,0.1,64
8,deepseek_r1d_qwen32b,proofwriter,4,0.0,0.3,64
9,deepseek_r1d_qwen32b,proofwriter,5,0.0,0.2,64



Refutation rate (head):


,model_key,dataset,depth,wrong_initial,flips_to_right,budget,refute_rate
0,deepseek_r1d_qwen32b,clutrr,2,10,0,64,0.0
1,deepseek_r1d_qwen32b,clutrr,3,10,0,64,0.0
2,deepseek_r1d_qwen32b,clutrr,4,10,0,64,0.0
3,deepseek_r1d_qwen32b,clutrr,5,10,0,64,0.0
4,deepseek_r1d_qwen32b,clutrr,6,10,0,64,0.0
5,deepseek_r1d_qwen32b,proofwriter,1,10,1,64,0.1
6,deepseek_r1d_qwen32b,proofwriter,2,10,2,64,0.2
7,deepseek_r1d_qwen32b,proofwriter,3,10,1,64,0.1
8,deepseek_r1d_qwen32b,proofwriter,4,10,3,64,0.3
9,deepseek_r1d_qwen32b,proofwriter,5,10,2,64,0.2


In [11]:
# STEP 6 — Inline tables for Colab (no downloads needed)
import os, glob, pandas as pd, numpy as np
from IPython.display import display, Markdown

tables_dir = RUN_DIR / "tables"
preds_dir  = RUN_DIR / "preds"
all_path   = RUN_DIR / "all_predictions.csv"

# ----------------- UI FILTERS (tweak these) -----------------
MODEL_FILTER   = None                     # e.g., ["llama31_8b","qwen25_7b"]; None = all
DATASET_FILTER = None                     # e.g., ["clutrr"] or ["proofwriter"]; None = both
DEPTH_FILTER   = None                     # e.g., [2,3,4] or [1,2,3]; None = all
MODES_FILTER   = None                     # e.g., ["direct","reasoning"]; None = all
SHOW_RAW       = False                    # True to show (truncated) raw model text per row
RAW_CHARS      = 280                      # truncate long raw to this many characters in table
MAX_ROWS       = 1000                     # cap number of rows printed per table to avoid giant outputs
# ------------------------------------------------------------

def _norm(s):
    return str(s).strip().lower() if isinstance(s, str) else str(s).lower()

def _truncate(s, n=280):
    if s is None or (isinstance(s, float) and np.isnan(s)): return ""
    s = str(s)
    return s if len(s) <= n else s[:n] + "…"

# -------- Load predictions (prefer combined; else concat preds/*.csv) --------
if all_path.exists():
    df = pd.read_csv(all_path)
else:
    files = sorted(glob.glob(str(preds_dir / "*.csv")))
    assert files, f"No predictions found in {preds_dir}. Have you run Step 4?"
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Normalize/compute columns
for col in ["depth","prompt_tokens","completion_tokens","est_cost_usd","d_hat","b_hat","budget"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df["correct"] = (df["gold"].map(_norm) == df["pred"].map(_norm)).astype(int)
df["ok"] = df["correct"].map({1:"✓", 0:"✗"})

# Apply filters
def apply_filters(dfin: pd.DataFrame) -> pd.DataFrame:
    d = dfin.copy()
    if MODEL_FILTER:
        d = d[d["model_key"].isin(MODEL_FILTER)]
    if DATASET_FILTER:
        d = d[d["dataset"].isin(DATASET_FILTER)]
    if DEPTH_FILTER:
        d = d[d["depth"].isin(DEPTH_FILTER)]
    if MODES_FILTER:
        d = d[d["mode"].isin(MODES_FILTER)]
    return d

df_f = apply_filters(df)

# ------------- Aggregates (if Step 5 ran we’ll show them too) ---------------
agg_tables = {}
for name in [
    "accuracy_by_depth.csv",
    "delta_reasoning.csv",
    "collapse_points.csv",
    "dcab_calibration.csv",
    "dcab_budget_efficiency.csv",
    "dcab_collapse_shift.csv",
    "av_vs_ta_accuracy.csv",
    "av_refutation_rate.csv",
    "spend_summary.csv",
]:
    p = tables_dir / name
    if p.exists():
        agg_tables[name] = pd.read_csv(p)

# ---------------------- Render: global summary header ------------------------
n_models = df_f["model_key"].nunique()
n_modes  = df_f["mode"].nunique()
n_rows   = len(df_f)
display(Markdown(f"### Results Overview  \n**Models:** {n_models}  |  **Modes:** {n_modes}  |  **Rows:** {n_rows}"))
if "spend_summary.csv" in agg_tables:
    display(Markdown("**Spend summary (from Step 5):**"))
    display(agg_tables["spend_summary.csv"].sort_values(["model_key","mode"]).reset_index(drop=True))

# ------------------ Render: Table‑2 style accuracy pivots --------------------
def make_accuracy_pivot(sub: pd.DataFrame) -> pd.DataFrame:
    acc = (sub.groupby(["model_key","mode","dataset","depth"])["correct"].mean()
           .reset_index())
    pv = (acc.pivot_table(index=["model_key","mode"], columns=["dataset","depth"], values="correct")
              .sort_index())
    return (pv * 100).round(1)

display(Markdown("## Accuracy by depth (Table‑2 view)"))
acc_pivot = make_accuracy_pivot(df_f[df_f["mode"].isin(["direct","reasoning"])])
display(acc_pivot.fillna(""))

# ------------------ Render: per‑model detailed tables ------------------------
def render_model_tables(model_key: str, sub_df: pd.DataFrame):
    display(Markdown(f"---\n## Model: `{model_key}`"))

    # Spend by mode
    spend = (sub_df.groupby("mode")
             .agg(n=("id","count"),
                  prompt_tokens=("prompt_tokens","sum"),
                  completion_tokens=("completion_tokens","sum"),
                  est_cost_usd=("est_cost_usd","sum"))
             .reset_index()
             .sort_values("mode"))
    display(Markdown("**Spend by mode:**"))
    display(spend)

    # Accuracy pivots for this model only
    display(Markdown("**Accuracy by depth (this model):**"))
    pv = make_accuracy_pivot(sub_df[sub_df["mode"].isin(["direct","reasoning"])])
    if model_key in pv.index.get_level_values(0):
        display(pv.loc[(model_key,), :].fillna(""))
    else:
        display(Markdown("_No Direct/Reasoning rows for this model._"))

    # Per‑item tables by mode
    for mode in sorted(sub_df["mode"].unique()):
        dmode = sub_df[sub_df["mode"] == mode].copy()
        if not len(dmode):
            continue
        cols = ["dataset","depth","id","gold","pred","ok","prompt_tokens","completion_tokens","est_cost_usd"]
        if "budget" in dmode.columns and dmode["budget"].notna().any():
            cols += ["budget"]
        if "d_hat" in dmode.columns and dmode["d_hat"].notna().any():
            cols += ["d_hat"]
        if "b_hat" in dmode.columns and dmode["b_hat"].notna().any():
            cols += ["b_hat"]
        if "changed_by_verify" in dmode.columns and dmode["changed_by_verify"].notna().any():
            cols += ["changed_by_verify"]
        if SHOW_RAW and "raw" in dmode.columns:
            dmode["raw_short"] = dmode["raw"].apply(lambda s: _truncate(s, RAW_CHARS))
            cols += ["raw_short"]

        to_show = dmode[cols].reset_index(drop=True)
        if len(to_show) > MAX_ROWS:
            display(Markdown(f"**{mode}** — showing first {MAX_ROWS} of {len(to_show)} rows"))
            display(to_show.head(MAX_ROWS))
        else:
            display(Markdown(f"**{mode}** — {len(to_show)} rows"))
            display(to_show)

# Drive the per‑model renderer
for mk in sorted(df_f["model_key"].unique()):
    render_model_tables(mk, df_f[df_f["model_key"] == mk])

# ---------------------- Optional: show Step‑5 aggregate tables ----------------
if agg_tables:
    display(Markdown("---\n## Aggregated tables (from Step 5)"))
    for name, tbl in agg_tables.items():
        display(Markdown(f"**{name}**"))
        display(tbl.head(MAX_ROWS))


### Results Overview  
**Models:** 5  |  **Modes:** 17  |  **Rows:** 8500

**Spend summary (from Step 5):**

,model_key,mode,total_prompt_tokens,total_completion_tokens,est_cost_usd
0,deepseek_r1d_qwen32b,av_128,22368,9600,0.0031
1,deepseek_r1d_qwen32b,av_256,22367,22124,0.0050
2,deepseek_r1d_qwen32b,av_64,22369,3200,0.0022
3,deepseek_r1d_qwen32b,av_answer@128,16621,3200,0.0017
4,deepseek_r1d_qwen32b,av_answer@256,16621,3200,0.0017
...,...,...,...,...,...
80,qwen25_7b,direct,17121,150,0.0007
81,qwen25_7b,reasoning,17821,161,0.0007
82,qwen25_7b,ta_128,17821,161,0.0007
83,qwen25_7b,ta_256,17821,161,0.0007


## Accuracy by depth (Table‑2 view)

dataset                        clutrr                         proofwriter  \
depth                               2     3     4     5     6           1   
model_key            mode                                                   
deepseek_r1d_qwen32b direct       0.0   0.0   0.0   0.0   0.0         0.0   
                     reasoning   10.0   0.0   0.0   0.0  10.0         0.0   
glm45_air            direct      10.0  10.0   0.0   0.0   0.0         0.0   
                     reasoning    0.0   0.0   0.0   0.0   0.0         0.0   
gpt5                 direct      10.0  10.0  10.0  10.0  10.0        50.0   
                     reasoning   20.0   0.0   0.0   0.0   0.0        20.0   
llama31_8b           direct      20.0  20.0  20.0  20.0  20.0        50.0   
                     reasoning   30.0  20.0  20.0  20.0  20.0        50.0   
qwen25_7b            direct      10.0   0.0   0.0  30.0   0.0        40.0   
                     reasoning   10.0   0.0  20.0   0.0   0.0        30.0   

dataset                                                 
depth                              2     3     4     5  
model_key            mode                               
deepseek_r1d_qwen32b direct      0.0   0.0   0.0   0.0  
                     reasoning   0.0   0.0   0.0   0.0  
glm45_air            direct      0.0   0.0   0.0   0.0  
                     reasoning   0.0   0.0   0.0   0.0  
gpt5                 direct     60.0  80.0  90.0  80.0  
                     reasoning  20.0  10.0   0.0   0.0  
llama31_8b           direct     20.0  50.0  40.0  60.0  
                     reasoning  20.0  50.0  50.0  50.0  
qwen25_7b            direct     30.0  40.0  30.0  60.0  
                     reasoning  20.0  40.0  40.0  50.0

...

,model_key,dataset,depth,acc_ta,acc_av,budget
0,deepseek_r1d_qwen32b,clutrr,2,0.1,0.0,64
1,deepseek_r1d_qwen32b,clutrr,3,0.0,0.0,64
2,deepseek_r1d_qwen32b,clutrr,4,0.0,0.0,64
3,deepseek_r1d_qwen32b,clutrr,5,0.0,0.0,64
4,deepseek_r1d_qwen32b,clutrr,6,0.1,0.0,64
...,...,...,...,...,...,...
145,qwen25_7b,proofwriter,1,0.3,0.2,256
146,qwen25_7b,proofwriter,2,0.2,0.2,256
147,qwen25_7b,proofwriter,3,0.4,0.2,256
148,qwen25_7b,proofwriter,4,0.4,0.4,256


**av_refutation_rate.csv**

,model_key,dataset,depth,wrong_initial,flips_to_right,budget,refute_rate
0,deepseek_r1d_qwen32b,clutrr,2,10,0,64,0.0
1,deepseek_r1d_qwen32b,clutrr,3,10,0,64,0.0
2,deepseek_r1d_qwen32b,clutrr,4,10,0,64,0.0
3,deepseek_r1d_qwen32b,clutrr,5,10,0,64,0.0
4,deepseek_r1d_qwen32b,clutrr,6,10,0,64,0.0
...,...,...,...,...,...,...,...
145,qwen25_7b,proofwriter,1,10,2,256,0.2
146,qwen25_7b,proofwriter,2,10,2,256,0.2
147,qwen25_7b,proofwriter,3,10,2,256,0.2
148,qwen25_7b,proofwriter,4,10,4,256,0.4


**spend_summary.csv**

,model_key,mode,total_prompt_tokens,total_completion_tokens,est_cost_usd
0,deepseek_r1d_qwen32b,av_128,22368,9600,0.0031
1,deepseek_r1d_qwen32b,av_256,22367,22124,0.0050
2,deepseek_r1d_qwen32b,av_64,22369,3200,0.0022
3,deepseek_r1d_qwen32b,av_answer@128,16621,3200,0.0017
4,deepseek_r1d_qwen32b,av_answer@256,16621,3200,0.0017
...,...,...,...,...,...
80,qwen25_7b,direct,17121,150,0.0007
81,qwen25_7b,reasoning,17821,161,0.0007
82,qwen25_7b,ta_128,17821,161,0.0007
83,qwen25_7b,ta_256,17821,161,0.0007
